# Plot Blackrock Contents

This notebook allows you to load any Blackrock file (`.nsX` or `.nev`), list all available channels (including broadband, analog, and digital), and interactively plot across the entire duration of the recording.

In [23]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import neo

# Set matplotlib to display inline or comment it out to be able to zoom in
%matplotlib qt

# --- CONFIG ---
# Replace with the actual path to your Blackrock file (.nsX or .nev)
# BR_FILE_PATH = r
# BR_FILE_PATH = r"Y:\Current Project Databases - NHP\2025 Cerebellum prosthesis\Nike\20260116_NRR_RW012\Blackrock\NRR_RW012_019.ns6"
# BR_FILE_PATH = r13
# BR_FILE_PATH = r14
# BR_FILE_PATH = r15
# BR_FILE_PATH = r16
# BR_FILE_PATH = r17
BR_FILE_PATH = r"Y:\Current Project Databases - NHP\2025 Cerebellum prosthesis\Nike\20260220_NRR_RW018\Blackrock\NRR_RW018_010.ns6"
# BR_FILE_PATH = r"Y:\Current Project Databases - NHP\2025 Cerebellum prosthesis\Nike\20260226_NRR_RW019\Blackrock\NRR_RW019_017.ns6"

### Step 1: Load File and List Channels

In [24]:
br_path = Path(BR_FILE_PATH)
if not br_path.exists() or not br_path.is_file():
    print(f"File not found: '{br_path}'")
    print("Please update the BR_FILE_PATH variable above.")
else:
    print(f"Loading Blackrock recording: {br_path.name}...")
    print("(This will automatically discover and load all aligned .nsX companion files!)")
    
    try:
        reader = neo.io.BlackrockIO(filename=str(br_path))
        bl = reader.read_block()
    except Exception as e:
        print(f"Failed to load Blackrock file(s): {e}")
        bl = None

    if bl is not None:
        # Gather all analog signals across all streams
        all_channels = []
        
        for seg in bl.segments:
            for asig in seg.analogsignals:
                fs = float(asig.sampling_rate.magnitude)
                dur = float(asig.t_stop - asig.t_start)
                
                # Neo includes names in the array annotations
                ch_names = asig.array_annotations.get('channel_names', [f"Ch{i}" for i in range(asig.shape[1])])
                
                for i in range(asig.shape[1]):
                    all_channels.append({
                        'name': ch_names[i],
                        'fs': fs,
                        'duration_sec': dur,
                        'signal': asig[:, i], # Reference to the neo object
                        'units': asig.units.dimensionality.string
                    })

        print("\n" + "="*40)
        print("       Recording Information       ")
        print("="*40)
        print(f"File:              {br_path.name}")
        print(f"Total Channels:    {len(all_channels)}")
        print("="*40)
        
        if len(all_channels) > 0:
            print("\nAvailable Channels (including auxiliary/analog):")
            for idx, ch_info in enumerate(all_channels):
                print(f"  [{idx:3d}] Name: {ch_info['name']:<15} | FS: {ch_info['fs']:>7.1f} Hz | Dur: {ch_info['duration_sec']:.1f}s")
        else:
            print("No channels found in this recording.")

Loading Blackrock recording: NRR_RW018_010.ns6...
(This will automatically discover and load all aligned .nsX companion files!)

       Recording Information       
File:              NRR_RW018_010.ns6
Total Channels:    143

Available Channels (including auxiliary/analog):
  [  0] Name: yaw_vel         | FS:  1000.0 Hz | Dur: 168.0s
  [  1] Name: gvpos           | FS:  1000.0 Hz | Dur: 168.0s
  [  2] Name: hhpos           | FS:  1000.0 Hz | Dur: 168.0s
  [  3] Name: hvpos           | FS:  1000.0 Hz | Dur: 168.0s
  [  4] Name: tcmd            | FS:  1000.0 Hz | Dur: 168.0s
  [  5] Name: vog_sync        | FS:  1000.0 Hz | Dur: 168.0s
  [  6] Name: heart_rate      | FS:  1000.0 Hz | Dur: 168.0s
  [  7] Name: hhv_filt        | FS:  1000.0 Hz | Dur: 168.0s
  [  8] Name: reward          | FS:  1000.0 Hz | Dur: 168.0s
  [  9] Name: hcmd1           | FS:  1000.0 Hz | Dur: 168.0s
  [ 10] Name: touchscreen_syn | FS:  1000.0 Hz | Dur: 168.0s
  [ 11] Name: squeeze         | FS:  1000.0 Hz | Dur: 

### Step 2: Plot Selected Channel
Set the `CHANNEL_INDEX` variable to the index (0, 1, 2...) of the channel you want to plot from the list above.

In [25]:
# Set which channel to plot based on the index printed above
CHANNEL_INDEX = 10  # Example: 12 is 'unit'

if 'all_channels' not in locals() or len(all_channels) == 0:
    print("Please run the loading cell first to load the channels.")
elif not (0 <= CHANNEL_INDEX < len(all_channels)):
    print(f"Invalid CHANNEL_INDEX: {CHANNEL_INDEX}. Must be between 0 and {len(all_channels)-1}.")
else:
    selected_ch = all_channels[CHANNEL_INDEX]
    print(f"Selected channel: [{CHANNEL_INDEX}] {selected_ch['name']}")
    print(f"Extracting full {selected_ch['duration_sec']:g} seconds of data...")
    
    # Extract trace
    trace = selected_ch['signal'].magnitude.flatten()
    fs = selected_ch['fs']
    num_samples = len(trace)
    t_axis = np.arange(num_samples) / fs
    
    # Set up matplotlib for plotting
    fig, ax = plt.subplots(1, 1, figsize=(16, 5))
        
    fig.suptitle(f"Blackrock Entire Trace: {br_path.name} | Channel: {selected_ch['name']}", fontsize=14)

    # Plot
    ax.plot(t_axis, trace, color='#1f77b4', linewidth=0.8)
    units_str = selected_ch['units']
    if units_str == 'dimensionless':
        ax.set_ylabel("Amplitude")
    else:
        ax.set_ylabel(f"Amplitude ({units_str})")
        
    ax.grid(True, alpha=0.3)
    
    # Despine
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.set_xlabel("Time (seconds)")
    plt.tight_layout()
    plt.show()

Selected channel: [10] touchscreen_syn
Extracting full 167.971 seconds of data...
